# Laboratorio ENEIC Notebook 01: Carga, armonización y calidad de datos

Este notebook cubre el **punto 1** del laboratorio:

1. Lectura de cada archivo Excel por separado (pandas + openpyxl), selección de columnas y conversión a Spark con **tipos explícitos**.
2. Identificación de procedencia (`archivo_origen`, `periodo_archivo`, `anio_archivo`, `trimestre_calendario`).
3. Unión de los cuatro archivos de 2025 con `unionByName`.
4. Diagnóstico de calidad: faltantes, unicidad de claves, validación de códigos contra el diccionario.
5. Filtros de la población analítica en un orden fijo, contabilizando exclusiones.
6. Escritura en Parquet del conjunto preparado de 2025 y del de 2026 **por separado**.


## 0. Preparación del ambiente

El Dockerfile del curso no incluye `openpyxl`, que pandas necesita para leer `.xlsx`. La siguiente celda lo instala solo si hace falta.

In [10]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
import openpyxl
print("openpyxl:", openpyxl.__version__)

openpyxl: 3.1.5


In [11]:
import os, gc, json, math, time, warnings
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

spark = (
    SparkSession.builder
    .appName("Lab ENEIC - 01 Carga y calidad")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("PySpark:", pyspark.__version__)
print("Spark  :", spark.version)
print("Java   :", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))

PySpark: 3.5.1
Spark  : 3.5.1
Java   : 17.0.20.1


### 0.1 Rutas y catálogo de archivos

Cada archivo se registra con su **período de procedencia**. Estas columnas se asignan a partir del archivo, **no** a partir de la columna `TRIMESTRE` (ver sección 2).

In [12]:
BASE_DIR   = Path(os.environ.get("LAB_BASE_DIR", "/opt/app"))
DATA_DIR   = BASE_DIR / "working_dir" / "eneic"
OUT_DIR    = BASE_DIR / "working_dir" / "eneic_parquet"
BRONZE_DIR = OUT_DIR / "bronze"          # un Parquet por archivo, columnas seleccionadas, sin filtrar
OUT_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS = pd.DataFrame(
    [
        ("2025T1", 2025, 1, "Personas_ENEIC_T1_2025.xlsx",                "Diccionario_Personas_ENEIC_I-2025.xlsx",  "Desarrollo y entrenamiento", 51588, 270),
        ("2025T2", 2025, 2, "Personas-ENEIC-T2-2025.xlsx",                "Diccionario_Personas_ENEIC_II-2025.xlsx", "Desarrollo y entrenamiento", 51167, 270),
        ("2025T3", 2025, 3, "Base-de-datos-Personas-ENEIC-III-2025.xlsx", "Diccionario-Personas-ENEIC-III-2025.xlsx","Desarrollo y entrenamiento", 51583, 270),
        ("2025T4", 2025, 4, "Base-de-datos-Personas-ENEIC-IV-2025.xlsx",  "Diccionario-Personas-ENEIC-IV-2025.xlsx", "Validación y entrenamiento final", 49338, 302),
        ("2026T1", 2026, 1, "Base-de-datos-Personas-ENEIC-I-2026.xlsx",   "Diccionario-Personas-ENEIC-I-2026.xlsx",  "Prueba final", 49843, 270),
    ],
    columns=["periodo_archivo", "anio_archivo", "trimestre_calendario", "archivo",
             "diccionario", "uso", "filas_esperadas", "columnas_esperadas"],
)
ARCHIVOS["existe_base"] = ARCHIVOS["archivo"].map(lambda f: (DATA_DIR / f).exists())
ARCHIVOS["existe_dicc"] = ARCHIVOS["diccionario"].map(lambda f: (DATA_DIR / f).exists())
display(ARCHIVOS)

PERIODOS       = ARCHIVOS.loc[ARCHIVOS["existe_base"], "periodo_archivo"].tolist()
PERIODOS_2025  = [p for p in PERIODOS if p.startswith("2025")]
PERIODOS_2026  = [p for p in PERIODOS if p.startswith("2026")]
print("Períodos disponibles:", PERIODOS)

,periodo_archivo,anio_archivo,trimestre_calendario,archivo,diccionario,uso,filas_esperadas,columnas_esperadas,existe_base,existe_dicc
0,2025T1,2025,1,Personas_ENEIC_T1_2025.xlsx,Diccionario_Personas_ENEIC_I-2025.xlsx,Desarrollo y entrenamiento,51588,270,True,True
1,2025T2,2025,2,Personas-ENEIC-T2-2025.xlsx,Diccionario_Personas_ENEIC_II-2025.xlsx,Desarrollo y entrenamiento,51167,270,True,True
2,2025T3,2025,3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,Diccionario-Personas-ENEIC-III-2025.xlsx,Desarrollo y entrenamiento,51583,270,True,True
3,2025T4,2025,4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,Diccionario-Personas-ENEIC-IV-2025.xlsx,Validación y entrenamiento final,49338,302,True,True
4,2026T1,2026,1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,Diccionario-Personas-ENEIC-I-2026.xlsx,Prueba final,49843,270,True,True


Períodos disponibles: ['2025T1', '2025T2', '2025T3', '2025T4', '2026T1']


### 0.2 Variables seleccionadas y funciones de normalización

Un mismo código puede llegar como número entero (`2`), como decimal (`2.0`, cuando la columna de Excel tiene vacíos) o como texto (`'2'`). Antes de unir los archivos se lleva todo a una representación única:

* **Variables categóricas** (`TRIMESTRE`, `DOMINIO`, `P03A03A`, `P05C16`, `OCUPADOS`) → texto sin decimales (`"2"`). Vacíos → `null`.
* **Variables numéricas** (`P02A03`, `P05C07A`, `P05C07B`, `P05H01A`, `P05D01`, `FACTOR`) → `double`. Texto no convertible → `null` y se contabiliza.
* **Identificadores** (`ANIO`, `NUM_HOGAR`, `NUM_PERSONA`) → enteros.

No se imputa ningún valor en esta etapa.

In [13]:
COLS_ID     = ["ANIO", "NUM_HOGAR", "NUM_PERSONA"]
COLS_CODIGO = ["TRIMESTRE", "DOMINIO", "P03A03A", "P05C16", "OCUPADOS"]
COLS_NUM    = ["FACTOR", "P02A03", "P05C07A", "P05C07B", "P05H01A", "P05D01"]
COLS_SEL    = ["ANIO", "TRIMESTRE", "DOMINIO", "NUM_HOGAR", "NUM_PERSONA", "FACTOR",
               "P02A03", "P03A03A", "P05C07A", "P05C07B", "P05C16", "P05D01", "P05H01A", "OCUPADOS"]

def es_nulo(v):
    return v is None or (isinstance(v, float) and math.isnan(v)) or (isinstance(v, str) and v.strip() == "")

def a_numero(v):
    # Convierte a float; devuelve None si está vacío o no es convertible.
    if es_nulo(v):
        return None
    try:
        return float(str(v).strip().replace(",", "")) if isinstance(v, str) else float(v)
    except (TypeError, ValueError):
        return None

def normalizar_codigo(v):
    # Código categórico -> texto canónico ('2', no '2.0'); vacío -> None.
    if es_nulo(v):
        return None
    x = a_numero(v)
    if x is not None and math.isfinite(x) and float(x).is_integer():
        return str(int(x))
    return str(v).strip()

def a_entero(v):
    x = a_numero(v)
    return int(x) if x is not None and math.isfinite(x) and x.is_integer() else None

SCHEMA_BRONZE = T.StructType(
    [
        T.StructField("archivo_origen", T.StringType(), False),
        T.StructField("periodo_archivo", T.StringType(), False),
        T.StructField("anio_archivo", T.IntegerType(), False),
        T.StructField("trimestre_calendario", T.IntegerType(), False),
        T.StructField("ANIO", T.IntegerType(), True),
        T.StructField("TRIMESTRE", T.StringType(), True),
        T.StructField("DOMINIO", T.StringType(), True),
        T.StructField("NUM_HOGAR", T.LongType(), True),
        T.StructField("NUM_PERSONA", T.IntegerType(), True),
        T.StructField("FACTOR", T.DoubleType(), True),
        T.StructField("P02A03", T.DoubleType(), True),
        T.StructField("P03A03A", T.StringType(), True),
        T.StructField("P05C07A", T.DoubleType(), True),
        T.StructField("P05C07B", T.DoubleType(), True),
        T.StructField("P05C16", T.StringType(), True),
        T.StructField("P05D01", T.DoubleType(), True),
        T.StructField("P05H01A", T.DoubleType(), True),
        T.StructField("OCUPADOS", T.StringType(), True),
        T.StructField("hash_fila", T.StringType(), False),   # huella de la fila ORIGINAL completa (todas las columnas)
    ]
)

### 0.3 Diccionarios de datos

Se leen las **etiquetas de valores** directamente de cada diccionario del INE para validar los códigos categóricos y verificar que no cambien entre períodos. También se leen las **posiciones** de las variables, que se usan para responder por qué IV de 2025 no puede apilarse por posición.

In [14]:
VARS_CATEGORICAS = ["DOMINIO", "P03A03A", "P05C16", "OCUPADOS", "P05C07B"]

def leer_diccionario(ruta):
    # Devuelve (posiciones {var: pos}, etiquetas {var: {codigo: etiqueta}}).
    wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    posiciones, etiquetas = {}, {}
    en_valores, actual = False, None
    for fila in ws.iter_rows(values_only=True):
        if not fila or all(c is None for c in fila):
            continue
        if fila[0] == "Valores de variable":
            en_valores = True
            continue
        if not en_valores:
            if fila[0] and a_entero(fila[1]) is not None:
                posiciones[str(fila[0]).strip()] = a_entero(fila[1])
            continue
        if fila[0] not in (None, "Valor"):
            actual = str(fila[0]).strip()
        codigo = normalizar_codigo(fila[1])
        etiqueta = next((c for c in fila[2:] if c not in (None, "")), None)
        if actual and codigo is not None and etiqueta is not None:
            etiquetas.setdefault(actual, {})[codigo] = str(etiqueta).strip().rstrip("?").strip()
    wb.close()
    return posiciones, etiquetas

DICC = {}
for _, r in ARCHIVOS[ARCHIVOS["existe_dicc"]].iterrows():
    DICC[r.periodo_archivo] = leer_diccionario(DATA_DIR / r.diccionario)

# ¿Son iguales los catálogos de códigos entre períodos?
ref = "2025T1"
comparacion = []
for var in VARS_CATEGORICAS:
    for p, (_, et) in DICC.items():
        comparacion.append({
            "variable": var, "periodo": p,
            "n_codigos": len(et.get(var, {})),
            "igual_a_2025T1": et.get(var, {}) == DICC[ref][1].get(var, {}),
        })
display(pd.DataFrame(comparacion).pivot(index="variable", columns="periodo", values="igual_a_2025T1"))

periodo,2025T1,2025T2,2025T3,2025T4,2026T1
variable,,,,,
DOMINIO,True,True,True,True,True
OCUPADOS,True,True,True,True,True
P03A03A,True,True,True,True,True
P05C07B,True,True,True,True,True
P05C16,True,True,True,True,True


In [15]:
_, ETQ = DICC[ref]
CAT_DOMINIO   = ETQ["DOMINIO"]
CAT_EDUCACION = ETQ["P03A03A"]
CAT_OCUPACION = ETQ["P05C16"]
CODIGOS_ASALARIADOS = ["1", "2", "3", "4"]

for nombre, cat in [("DOMINIO", CAT_DOMINIO), ("P03A03A (nivel educativo)", CAT_EDUCACION),
                    ("P05C16 (categoría ocupacional)", CAT_OCUPACION), ("OCUPADOS", ETQ["OCUPADOS"])]:
    print(f"{nombre}: {cat}")

DOMINIO: {'1': 'Urbano Metropolitano', '2': 'Resto Urbano', '3': 'Rural Nacional'}
P03A03A (nivel educativo): {'0': 'NINGUNO', '1': 'PREPRIMARIA', '2': 'PRIMARIA', '3': 'BÁSICO', '4': 'DIVERSIFICADO', '5': 'SUPERIOR', '6': 'MAESTRÍA', '7': 'DOCTORADO'}
P05C16 (categoría ocupacional): {'1': 'Empleado de gobierno', '2': 'Empleado de empresa privada', '3': 'Empleado jornalero o peón', '4': 'Del servicio doméstico', '5': 'Trabajador por cuenta propia NO agrícola', '6': 'Patrón empleador (a) socio (a) NO agrícola', '7': 'Trabajador por cuenta propia agrícola', '8': 'Patrón empleador (a) socio (a) agrícola', '9': 'Trabajador No remunerado'}
OCUPADOS: {'1': 'Población ocupada'}


**Lectura de los diccionarios.** Los catálogos de `DOMINIO`, `P03A03A`, `P05C16`, `OCUPADOS` y `P05C07B` tienen el mismo contenido en todos los períodos, por lo que un único catálogo sirve para validar los cinco archivos. Dos detalles relevantes:

* `P03A03A = 0` significa **"Ninguno"**: es un nivel educativo válido, no un faltante.
* `OCUPADOS` solo tiene la etiqueta `1 = Población ocupada`; las personas no ocupadas quedan en blanco. Por eso un vacío en `OCUPADOS` no es "no respuesta", sino "no pertenece a la población ocupada".
* `P05C07B` (meses de antigüedad) tiene como códigos válidos 0 a 11, lo que respalda el criterio de “componente de meses entero entre 0 y 11”.
* En el diccionario de IV 2025 los códigos están guardados como **texto** y en los demás como **número**; la función `normalizar_codigo` permite compararlos correctamente.

## 1. Carga de cada archivo y conversión a Parquet

Cada Excel se procesa **individualmente**: se lee con pandas, se calcula una huella (`hash_fila`) de la fila original completa, se seleccionan y normalizan las 14 columnas, se crea el DataFrame de Spark con el esquema explícito y se escribe en Parquet. Luego se libera la memoria de pandas antes de pasar al siguiente archivo.

Si el Parquet de un archivo ya existe, se reutiliza (poner `RECONVERTIR = True` para forzar la lectura del Excel de nuevo).

In [16]:
RECONVERTIR = False

def tipo_origen(v):
    return "vacío" if es_nulo(v) else type(v).__name__

def convertir_archivo(r):
    destino = BRONZE_DIR / r.periodo_archivo
    meta_path = BRONZE_DIR / f"{r.periodo_archivo}_meta.json"
    if destino.exists() and meta_path.exists() and not RECONVERTIR:
        print(f"{r.periodo_archivo}: Parquet existente reutilizado")
        return json.loads(meta_path.read_text())

    t0 = time.time()
    pdf = pd.read_excel(DATA_DIR / r.archivo, sheet_name=0, dtype=object)
    faltan = [c for c in COLS_SEL if c not in pdf.columns]
    if faltan:
        raise ValueError(f"{r.archivo}: faltan columnas {faltan}")

    meta = {
        "periodo_archivo": r.periodo_archivo,
        "archivo_origen": r.archivo,
        "filas": int(pdf.shape[0]),
        "columnas": int(pdf.shape[1]),
        "lista_columnas": [str(c) for c in pdf.columns],
        "tipos_origen": {c: pdf[c].map(tipo_origen).value_counts().to_dict() for c in COLS_SEL},
        "no_convertibles": {},
    }

    sel = pd.DataFrame(index=pdf.index)
    sel["archivo_origen"] = r.archivo
    sel["periodo_archivo"] = r.periodo_archivo
    sel["anio_archivo"] = int(r.anio_archivo)
    sel["trimestre_calendario"] = int(r.trimestre_calendario)
    for c in COLS_ID:
        sel[c] = pdf[c].map(a_entero)
    for c in COLS_CODIGO:
        sel[c] = pdf[c].map(normalizar_codigo)
    for c in COLS_NUM:
        sel[c] = pdf[c].map(a_numero)
    for c in COLS_ID + COLS_NUM:
        convertido = sel[c]
        meta["no_convertibles"][c] = int(((~pdf[c].map(es_nulo)) & convertido.isna()).sum())
    sel["hash_fila"] = pd.util.hash_pandas_object(pdf.astype(str), index=False).astype(str)

    columnas = [f.name for f in SCHEMA_BRONZE.fields]
    filas = sel[columnas].astype(object).where(sel[columnas].notna(), None).values.tolist()
    sdf = spark.createDataFrame(filas, schema=SCHEMA_BRONZE)
    sdf.write.mode("overwrite").parquet(str(destino))

    meta_path.write_text(json.dumps(meta, ensure_ascii=False))
    del pdf, sel, filas
    gc.collect()
    print(f"{r.periodo_archivo}: {meta['filas']:,} filas x {meta['columnas']} columnas "
          f"convertidas en {time.time() - t0:.0f} s")
    return meta

META = {}
for _, r in ARCHIVOS[ARCHIVOS["existe_base"]].iterrows():
    META[r.periodo_archivo] = convertir_archivo(r)

2025T1: 51,588 filas x 270 columnas convertidas en 106 s


2025T2: 51,167 filas x 270 columnas convertidas en 44 s
2025T3: 51,583 filas x 270 columnas convertidas en 102 s


2025T4: 49,338 filas x 302 columnas convertidas en 112 s


2026T1: 49,843 filas x 270 columnas convertidas en 100 s


### 1.1 Verificación de dimensiones y tipos de origen

In [17]:
dim = pd.DataFrame([
    {"periodo_archivo": p, "archivo_origen": m["archivo_origen"], "filas": m["filas"], "columnas": m["columnas"]}
    for p, m in META.items()
]).merge(ARCHIVOS[["periodo_archivo", "filas_esperadas", "columnas_esperadas"]], on="periodo_archivo")
dim["coincide"] = (dim.filas == dim.filas_esperadas) & (dim.columnas == dim.columnas_esperadas)
display(dim)

tipos = pd.DataFrame({
    p: {c: ", ".join(f"{k}:{v:,}" for k, v in m["tipos_origen"][c].items()) for c in COLS_SEL}
    for p, m in META.items()
})
print("Tipos de Python con los que llega cada columna desde Excel (tipo:cantidad):")
display(tipos)

no_conv = pd.DataFrame({p: m["no_convertibles"] for p, m in META.items()})
print("Valores no vacíos que no pudieron convertirse a número (deben ser 0):")
display(no_conv)

,periodo_archivo,archivo_origen,filas,columnas,filas_esperadas,columnas_esperadas,coincide
0,2025T1,Personas_ENEIC_T1_2025.xlsx,51588,270,51588,270,True
1,2025T2,Personas-ENEIC-T2-2025.xlsx,51167,270,51167,270,True
2,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,51583,270,51583,270,True
3,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,49338,302,49338,302,True
4,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,49843,270,49843,270,True


Tipos de Python con los que llega cada columna desde Excel (tipo:cantidad):


,2025T1,2025T2,2025T3,2025T4,2026T1
ANIO,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
TRIMESTRE,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
DOMINIO,"int:51,588","str:51,167","int:51,583","int:49,338","int:49,843"
NUM_HOGAR,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
NUM_PERSONA,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
FACTOR,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
P02A03,"int:51,588","int:51,167","int:51,583","int:49,338","int:49,843"
P03A03A,"int:44,660, vacío:6,928","str:44,489, vacío:6,678","int:45,035, vacío:6,548","int:43,148, vacío:6,190","int:43,832, vacío:6,011"
P05C07A,"vacío:29,315, int:22,273","vacío:28,824, int:22,343","vacío:29,210, int:22,373","vacío:28,105, int:21,233","vacío:28,109, int:21,734"
P05C07B,"vacío:29,315, int:22,273","vacío:28,824, str:22,343","vacío:29,210, int:22,373","vacío:28,105, int:21,233","vacío:28,109, int:21,734"


Valores no vacíos que no pudieron convertirse a número (deben ser 0):


,2025T1,2025T2,2025T3,2025T4,2026T1
ANIO,0,0,0,0,0
NUM_HOGAR,0,0,0,0,0
NUM_PERSONA,0,0,0,0,0
FACTOR,0,0,0,0,0
P02A03,0,0,0,0,0
P05C07A,0,0,0,0,0
P05C07B,0,0,0,0,0
P05H01A,0,0,0,0,0
P05D01,0,0,0,0,0


**Interpretación.** Las dimensiones coinciden con las publicadas (tabla del enunciado). La tabla de tipos muestra el problema de homologación: en II 2025, `DOMINIO`, `P03A03A`, `P05C16`, `P05C07B` y `OCUPADOS` llegan como **texto** (`str`), mientras que en los demás archivos llegan como **números** (`int`, o `float` cuando la columna tiene vacíos, porque pandas representa el vacío como `NaN`). Si se unieran sin normalizar, `"2"` y `2.0` serían valores distintos para el mismo código. Tras la normalización todos los códigos son texto canónico y no hubo valores no convertibles.

## 2. Unión de los archivos 2025 con `unionByName`

In [18]:
BRONZE = {p: spark.read.parquet(str(BRONZE_DIR / p)) for p in PERIODOS}

df2025_raw = reduce(lambda a, b: a.unionByName(b), [BRONZE[p] for p in PERIODOS_2025]).cache()
df2026_raw = BRONZE[PERIODOS_2026[0]].cache()

print("Registros 2025 (unión):", f"{df2025_raw.count():,}")
print("Registros 2026        :", f"{df2026_raw.count():,}")
df2025_raw.printSchema()

Registros 2025 (unión): 203,676
Registros 2026        : 49,843
root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: string (nullable = true)
 |-- DOMINIO: string (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- FACTOR: double (nullable = true)
 |-- P02A03: double (nullable = true)
 |-- P03A03A: string (nullable = true)
 |-- P05C07A: double (nullable = true)
 |-- P05C07B: double (nullable = true)
 |-- P05C16: string (nullable = true)
 |-- P05D01: double (nullable = true)
 |-- P05H01A: double (nullable = true)
 |-- OCUPADOS: string (nullable = true)
 |-- hash_fila: string (nullable = true)



In [19]:
df2025_raw.select(
    "archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario", *COLS_SEL
).show(5, truncate=False)

+---------------------------+---------------+------------+--------------------+----+---------+-------+---------+-----------+------+------+-------+-------+-------+------+------+-------+--------+
|archivo_origen             |periodo_archivo|anio_archivo|trimestre_calendario|ANIO|TRIMESTRE|DOMINIO|NUM_HOGAR|NUM_PERSONA|FACTOR|P02A03|P03A03A|P05C07A|P05C07B|P05C16|P05D01|P05H01A|OCUPADOS|
+---------------------------+---------------+------------+--------------------+----+---------+-------+---------+-----------+------+------+-------+-------+-------+------+------+-------+--------+
|Personas_ENEIC_T1_2025.xlsx|2025T1         |2025        |1                   |2025|2        |2      |12647    |1          |519.0 |80.0  |2      |NULL   |NULL   |NULL  |NULL  |NULL   |NULL    |
|Personas_ENEIC_T1_2025.xlsx|2025T1         |2025        |1                   |2025|2        |2      |12647    |6          |519.0 |18.0  |4      |NULL   |NULL   |NULL  |NULL  |NULL   |NULL    |
|Personas_ENEIC_T1_2025.xlsx|2

### 2.1 Identificación del período: `TRIMESTRE` original vs. archivo de procedencia

In [20]:
(df2025_raw.unionByName(df2026_raw)
    .groupBy("periodo_archivo").pivot("TRIMESTRE").count()
    .orderBy("periodo_archivo").fillna(0).show())

[Stage 29:============================================>          (48 + 12) / 60]

+---------------+-----+-----+-----+-----+-----+
|periodo_archivo|    2|    3|    4|    5|    6|
+---------------+-----+-----+-----+-----+-----+
|         2025T1|51588|    0|    0|    0|    0|
|         2025T2|  175|50992|    0|    0|    0|
|         2025T3|    0|    0|51583|    0|    0|
|         2025T4|    0|    0|    0|49338|    0|
|         2026T1|    0|    0|    0|    0|49843|
+---------------+-----+-----+-----+-----+-----+



**Interpretación.** La columna `TRIMESTRE` original no corresponde al trimestre calendario: en I 2025 vale 2, en IV 2025 vale 5 y en I 2026 vale 6 (es un número correlativo de levantamiento). Además, II 2025 mezcla 175 registros con `TRIMESTRE = 2`. Restar 1 funcionaría para la mayoría de registros, pero dejaría esos 175 asignados al primer trimestre aunque pertenecen al archivo publicado de II 2025. Por eso `periodo_archivo`, `anio_archivo` y `trimestre_calendario` se asignan **desde el archivo** y `TRIMESTRE` se conserva tal cual, solo para auditoría.

### 2.2 ¿Por qué IV de 2025 no puede apilarse por posición?

In [21]:
cols = {p: m["lista_columnas"] for p, m in META.items()}
ref_cols = cols["2025T1"]
iv_cols = cols.get("2025T4", [])
solo_iv  = [c for c in iv_cols if c not in ref_cols]
solo_ref = [c for c in ref_cols if c not in iv_cols]
print(f"Columnas en IV 2025 que no están en I 2025 ({len(solo_iv)}):", solo_iv)
print(f"Columnas en I 2025 que no están en IV 2025 ({len(solo_ref)}):", solo_ref)

posiciones = pd.DataFrame({p: {c: cols[p].index(c) + 1 for c in COLS_SEL} for p in cols})
print("\nPosición (1 = primera columna) de las variables seleccionadas en cada archivo:")
display(posiciones)

# Qué pasaría si se apilara por posición: la columna 13 de IV no es la edad
if iv_cols:
    print("Columna #13 en I 2025:", ref_cols[12], "| columna #13 en IV 2025:", iv_cols[12])

Columnas en IV 2025 que no están en I 2025 (42): ['P07A01A', 'P07A01B', 'P07A01C', 'P07A02A', 'P07A02B', 'P07A02C', 'P07A03A', 'P07A03B', 'P07A03C', 'P07A04A', 'P07A04B', 'P07A04C', 'P08A01A', 'P08A01B', 'P08A01C', 'P08A02A', 'P08A02B', 'P08A02C', 'P08B01A', 'P08B01B', 'P08B01C', 'P08B02A', 'P08B02B', 'P08B02C', 'P08C01A', 'P08C01B', 'P08C01C', 'P08D01A', 'P08D01B', 'P08D01C', 'P08E01A', 'P08E01B', 'P08E01C', 'P08E02A', 'P08E02B', 'P08E02C', 'P08E03A', 'P08E03B', 'P08E03C', 'P08F01A', 'P08F01B', 'P08F01C']
Columnas en I 2025 que no están en IV 2025 (10): ['P02A01C', 'P02A01D', 'P02A01G', 'P02A01I', 'P05C01A', 'P05C02A', 'P05C04A', 'P05G01A', 'P05G02A', 'P05G04A']

Posición (1 = primera columna) de las variables seleccionadas en cada archivo:


,2025T1,2025T2,2025T3,2025T4,2026T1
ANIO,1,1,1,1,1
TRIMESTRE,2,2,2,2,2
DOMINIO,3,3,3,3,3
NUM_HOGAR,4,4,4,4,4
NUM_PERSONA,7,7,7,7,7
FACTOR,5,5,5,5,5
P02A03,13,13,13,9,13
P03A03A,33,33,33,29,33
P05C07A,68,68,68,61,68
P05C07B,69,69,69,62,69


Columna #13 en I 2025: P02A03 | columna #13 en IV 2025: P02A06A


**Respuesta.** IV 2025 tiene **302 columnas** en lugar de 270: el INE eliminó algunas preguntas y agregó otras (listas arriba). Como consecuencia, las mismas variables ocupan **posiciones distintas** (por ejemplo, la edad `P02A03` está en la columna 13 en los otros archivos y en la 9 en IV; `P05D01` pasa de la 105 a la 98). Un `union()` por posición pegaría la columna 13 de IV debajo de la edad de los demás trimestres sin dar ningún error, lo que corrompería silenciosamente los datos. `unionByName` alinea por nombre de columna y, al haber seleccionado primero las mismas 14 columnas en todos los archivos, garantiza que cada variable se apile consigo misma.

## 3. Faltantes por variable seleccionada (antes de filtros)

In [22]:
def tabla_faltantes(df, columnas):
    total = df.count()
    exprs = []
    for c in columnas:
        tipo = dict(df.dtypes)[c]
        cond = F.col(c).isNull() | F.isnan(F.col(c)) if tipo == "double" else F.col(c).isNull()
        exprs.append(F.sum(cond.cast("int")).alias(c))
    fila = df.agg(*exprs).collect()[0].asDict()
    out = pd.DataFrame({"faltantes": fila}).rename_axis("variable")
    out["porcentaje"] = (100 * out["faltantes"] / total).round(2)
    out["registros"] = total
    return out

VARS_FALTANTES = ["ANIO", "TRIMESTRE", "DOMINIO", "NUM_HOGAR", "NUM_PERSONA", "FACTOR",
                  "P02A03", "P03A03A", "P05C07A", "P05C07B", "P05C16", "P05D01", "P05H01A", "OCUPADOS"]

falt_2025 = tabla_faltantes(df2025_raw, VARS_FALTANTES)
falt_2026 = tabla_faltantes(df2026_raw, VARS_FALTANTES)
display(pd.concat({"2025 (unión)": falt_2025, "2026T1": falt_2026}, axis=1))

2025 (unión)                         2026T1                     
               faltantes porcentaje registros faltantes porcentaje registros
variable                                                                    
ANIO                   0       0.00    203676         0       0.00     49843
TRIMESTRE              0       0.00    203676         0       0.00     49843
DOMINIO                0       0.00    203676         0       0.00     49843
NUM_HOGAR              0       0.00    203676         0       0.00     49843
NUM_PERSONA            0       0.00    203676         0       0.00     49843
FACTOR                 0       0.00    203676         0       0.00     49843
P02A03                 0       0.00    203676         0       0.00     49843
P03A03A            26344      12.93    203676      6011      12.06     49843
P05C07A           115454      56.69    203676     28109      56.40     49843
P05C07B           115454      56.69    203676     28109      56.40     49843
P05C16            115454      56.69    203676     28109      56.40     49843
P05D01            150651      73.97    203676     36585      73.40     49843
P05H01A           115454      56.69    203676     28109      56.40     49843
OCUPADOS          115454      56.69    203676     28109      56.40     49843

In [23]:
# Porcentaje de faltantes por archivo (misma lógica, desagregado)
display(pd.DataFrame({p: tabla_faltantes(BRONZE[p], VARS_FALTANTES)["porcentaje"] for p in PERIODOS}))

,2025T1,2025T2,2025T3,2025T4,2026T1
variable,,,,,
ANIO,0.00,0.00,0.00,0.00,0.00
TRIMESTRE,0.00,0.00,0.00,0.00,0.00
DOMINIO,0.00,0.00,0.00,0.00,0.00
NUM_HOGAR,0.00,0.00,0.00,0.00,0.00
NUM_PERSONA,0.00,0.00,0.00,0.00,0.00
FACTOR,0.00,0.00,0.00,0.00,0.00
P02A03,0.00,0.00,0.00,0.00,0.00
P03A03A,13.43,13.05,12.69,12.55,12.06
P05C07A,56.83,56.33,56.63,56.96,56.40


### 3.1 Faltante estructural vs. no respuesta

Los porcentajes altos de la tabla anterior no significan que los encuestados no respondieron. Se separan los vacíos según si la pregunta **correspondía** a la persona:

In [24]:
es_ocupado    = F.col("OCUPADOS") == "1"
es_asalariado = F.col("P05C16").isin(CODIGOS_ASALARIADOS)

diag = df2025_raw.select(
    F.when(F.col("P02A03") < 15, "1. Menor de 15 años")
     .when(~F.coalesce(es_ocupado, F.lit(False)), "2. 15+ no ocupado")
     .when(~F.coalesce(es_asalariado, F.lit(False)), "3. 15+ ocupado no asalariado")
     .otherwise("4. 15+ ocupado asalariado").alias("grupo"),
    *[F.col(c) for c in ["P03A03A", "P05C16", "P05C07A", "P05H01A", "P05D01"]]
)
res = (diag.groupBy("grupo")
       .agg(F.count("*").alias("registros"),
            *[F.round(100 * F.avg((F.col(c).isNull() | (F.isnan(c) if c in COLS_NUM else F.lit(False))).cast("int")), 2).alias(f"%falt_{c}")
              for c in ["P03A03A", "P05C16", "P05C07A", "P05H01A", "P05D01"]])
       .orderBy("grupo"))
res.show(truncate=False)

(df2025_raw.filter(F.col("P03A03A").isNull())
    .agg(F.min("P02A03").alias("edad_min"), F.max("P02A03").alias("edad_max"), F.count("*").alias("n"))
    .show())

+----------------------------+---------+-------------+------------+-------------+-------------+------------+
|grupo                       |registros|%falt_P03A03A|%falt_P05C16|%falt_P05C07A|%falt_P05H01A|%falt_P05D01|
+----------------------------+---------+-------------+------------+-------------+-------------+------------+
|1. Menor de 15 años         |62886    |41.89        |100.0       |100.0        |100.0        |100.0       |
|2. 15+ no ocupado           |52568    |0.0          |100.0       |100.0        |100.0        |100.0       |
|3. 15+ ocupado no asalariado|35197    |0.0          |0.0         |0.0          |0.0          |100.0       |
|4. 15+ ocupado asalariado   |53025    |0.0          |0.0         |0.0          |0.0          |0.0         |
+----------------------------+---------+-------------+------------+-------------+-------------+------------+

+--------+--------+-----+
|edad_min|edad_max|    n|
+--------+--------+-----+
|     0.0|     6.0|26344|
+--------+--------+----

**Respuesta: ¿qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**

* **Ausente por flujo del cuestionario (estructural o "no aplica").** La boleta tiene saltos: las preguntas del capítulo de empleo (`P05C16`, antigüedad, horas) solo se hacen a personas ocupadas, y el salario `P05D01` solo a quienes trabajan por sueldo o salario. La tabla muestra que **el 100 %** de los vacíos en `P05C16`, `P05C07A`, `P05H01A` y `P05D01` corresponde a menores de 15 años, a personas no ocupadas o a ocupados no asalariados. Del mismo modo, `P03A03A` solo está vacío para niños de 0 a 6 años, a quienes no se pregunta el nivel educativo. Estos vacíos **no son información perdida**: el valor no existe para esa persona, y lo correcto es excluirla de la población de estudio, no imputar.
* **Respuesta no registrada (no respuesta).** La pregunta sí correspondía pero el dato quedó vacío (se negó a responder, no sabía, error de captura) o con un código de “no sabe / no responde”. Solo este tipo implica pérdida de información y posible sesgo (por ejemplo, si quienes ganan más responden menos).

En las bases publicadas, la fila “15+ ocupado asalariado” tiene **0 % de faltantes** en todas las variables, incluido el salario, y el diccionario no define códigos especiales de no respuesta para `P05D01`, `P05H01A` ni la antigüedad. Es decir, en los archivos publicados **no se observa no respuesta** entre asalariados: los vacíos son todos estructurales. Esto sugiere que el INE depura o trata la no respuesta antes de publicar, lo cual debe mencionarse como limitación: no podemos saber cuántos asalariados no declararon su salario originalmente, ni cómo se resolvió ese caso. Aun así, el filtro 4 se mantiene para que el procedimiento sea válido con cualquier archivo.

## 4. Unicidad de la clave `periodo_archivo` + `NUM_HOGAR` + `NUM_PERSONA`

In [25]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]

def revisar_unicidad(df, nombre):
    total = df.count()
    distintas = df.select(*CLAVE).distinct().count()
    dup = df.groupBy(*CLAVE).agg(F.count("*").alias("n"), F.countDistinct("hash_fila").alias("versiones")).filter("n > 1")
    n_dup = dup.count()
    print(f"{nombre}: {total:,} registros | {distintas:,} claves distintas | claves repetidas: {n_dup:,} | "
          f"clave nula: {df.filter(F.col('NUM_HOGAR').isNull() | F.col('NUM_PERSONA').isNull()).count()}")
    if n_dup:
        clasif = dup.withColumn("tipo", F.when(F.col("versiones") == 1, "repetición exacta").otherwise("registros en conflicto"))
        clasif.groupBy("periodo_archivo", "tipo").count().show()
        clasif.orderBy(F.desc("n")).show(10)
    return n_dup

dups_2025 = revisar_unicidad(df2025_raw, "2025 (sin filtros)")
dups_2026 = revisar_unicidad(df2026_raw, "2026 (sin filtros)")

2025 (sin filtros): 203,676 registros | 203,676 claves distintas | claves repetidas: 0 | clave nula: 0
2026 (sin filtros): 49,843 registros | 49,843 claves distintas | claves repetidas: 0 | clave nula: 0


**Interpretación.** La combinación `periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA` es **única** en todos los archivos: no hay claves repetidas ni nulas. El código deja preparada la investigación de duplicados: si apareciera una clave repetida, se compararía la huella `hash_fila` de la fila **original completa**; una sola huella indicaría una repetición exacta (mismo registro cargado dos veces), y huellas distintas indicarían **registros en conflicto** (dos personas con la misma clave o una captura corregida). En ningún caso se usa `dropDuplicates()`, porque ocultaría cuál de los dos casos ocurre.

Nótese que la clave **incluye el período**. Sin él, la misma persona aparece legítimamente varias veces, como se muestra a continuación.

In [26]:
# ¿Cuántas personas (NUM_HOGAR, NUM_PERSONA) aparecen en más de un período de 2025?
apariciones = (df2025_raw.groupBy("NUM_HOGAR", "NUM_PERSONA")
               .agg(F.countDistinct("periodo_archivo").alias("periodos")))
apariciones.groupBy("periodos").count().orderBy("periodos").show()

# Evidencia de que es la misma persona: diferencia de edad entre dos períodos consecutivos
pares = [(a, b) for a, b in zip(PERIODOS_2025, PERIODOS_2025[1:])]
for a, b in pares:
    ja = BRONZE[a].select("NUM_HOGAR", "NUM_PERSONA", F.col("P02A03").alias("edad_a"))
    jb = BRONZE[b].select("NUM_HOGAR", "NUM_PERSONA", F.col("P02A03").alias("edad_b"))
    j = ja.join(jb, ["NUM_HOGAR", "NUM_PERSONA"])
    r = j.agg(F.count("*").alias("en_ambos"),
              F.round(100 * F.avg((F.abs(F.col("edad_b") - F.col("edad_a")) <= 1).cast("int")), 1).alias("pct_edad_consistente")).collect()[0]
    print(f"{a} vs {b}: {r.en_ambos:,} claves en ambos períodos; {r.pct_edad_consistente}% con diferencia de edad de 0 o 1 año")

+--------+-----+
|periodos|count|
+--------+-----+
|       1|31273|
|       2|27771|
|       3|16279|
|       4|17006|
+--------+-----+

2025T1 vs 2025T2: 37,008 claves en ambos períodos; 99.7% con diferencia de edad de 0 o 1 año
2025T2 vs 2025T3: 37,132 claves en ambos períodos; 99.7% con diferencia de edad de 0 o 1 año
2025T3 vs 2025T4: 36,893 claves en ambos períodos; 99.8% con diferencia de edad de 0 o 1 año


**Respuesta: ¿por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

La ENEIC es un panel con **rotación**: una parte de los hogares se entrevista en trimestres sucesivos. Una misma persona en I y II de 2025 no es un error de carga, sino **dos observaciones válidas en momentos distintos**, con salario, horas o antigüedad que pueden haber cambiado. La edad casi siempre coincide o aumenta en un año, lo que confirma que son las mismas personas y no colisiones de identificadores. Eliminar la segunda aparición:

* borraría información real de uno de los trimestres y sesgaría los conteos y medianas por período (cada trimestre quedaría sin los hogares que continúan en el panel);
* rompería la comparabilidad entre trimestres, que es justo lo que se analiza en el punto 2.

La unidad de análisis es **persona-período**, y por eso la clave de unicidad incluye `periodo_archivo`. La repetición de personas sí tiene una consecuencia metodológica que se tomará en cuenta más adelante: al validar modelos, las observaciones de una misma persona no son independientes. Por eso el número de filas **no** equivale al número de personas distintas.

## 5. Filtros de la población analítica (orden fijo)

Los filtros se aplican **siempre en el mismo orden**, a 2025 y a 2026. En cada paso se separan dos tipos de exclusión:

* **no evaluable**: el dato necesario falta o no es finito, por lo que no puede evaluarse el criterio;
* **no cumple**: el dato existe, pero no satisface el criterio.

| Paso | Criterio |
|---|---|
| 1 | Edad finita y ≥ 15 |
| 2 | Ocupado (`OCUPADOS = 1`) |
| 3 | Asalariado (`P05C16` ∈ {1, 2, 3, 4}) |
| 4 | Salario `P05D01` numérico, finito y > 0 |
| 5 | Antigüedad evaluable: años ≥ 0 y meses entero entre 0 y 11 |
| 6 | Antigüedad calculada ≤ edad |
| 7 | Horas habituales > 0 y ≤ 168 |

In [27]:
def es_finito(c):
    c = F.col(c)
    return c.isNotNull() & ~F.isnan(c) & (F.abs(c) != F.lit(float("inf")))

antig = F.col("P05C07A") + F.col("P05C07B") / 12

PASOS = [
    ("1. Edad finita y >= 15",           es_finito("P02A03"),                          F.col("P02A03") >= 15),
    ("2. Ocupado (OCUPADOS = 1)",        F.lit(True),                                  F.coalesce(F.col("OCUPADOS") == "1", F.lit(False))),
    ("3. Asalariado (P05C16 en 1-4)",    F.col("P05C16").isNotNull(),                  F.col("P05C16").isin(CODIGOS_ASALARIADOS)),
    ("4. Salario finito y > 0",          es_finito("P05D01"),                          F.col("P05D01") > 0),
    ("5. Antigüedad válida (años>=0, meses 0-11 enteros)",
                                         es_finito("P05C07A") & es_finito("P05C07B"),
                                         (F.col("P05C07A") >= 0) & (F.col("P05C07B") >= 0) & (F.col("P05C07B") <= 11)
                                         & (F.col("P05C07B") == F.floor("P05C07B"))),
    ("6. Antigüedad <= edad",            F.lit(True),                                  antig <= F.col("P02A03")),
    ("7. Horas habituales en (0, 168]",  es_finito("P05H01A"),                         (F.col("P05H01A") > 0) & (F.col("P05H01A") <= 168)),
]

def aplicar_filtros(df):
    actual = df
    filas = []
    for nombre, evaluable, cumple in PASOS:
        marcado = (actual
                   .withColumn("_eval", F.coalesce(evaluable, F.lit(False)))
                   .withColumn("_ok", F.col("_eval") & F.coalesce(cumple, F.lit(False))))
        res = (marcado.groupBy("periodo_archivo")
               .agg(F.count("*").alias("entran"),
                    F.sum((~F.col("_eval")).cast("int")).alias("excl_no_evaluable"),
                    F.sum((F.col("_eval") & ~F.col("_ok")).cast("int")).alias("excl_no_cumple"),
                    F.sum(F.col("_ok").cast("int")).alias("quedan"))
               .toPandas())
        res.insert(0, "paso", nombre)
        filas.append(res)
        actual = marcado.filter("_ok").drop("_eval", "_ok").cache()
    tabla = pd.concat(filas, ignore_index=True)
    tabla["excluidos"] = tabla["excl_no_evaluable"] + tabla["excl_no_cumple"]
    return actual, tabla

df2025_filt, pasos_2025 = aplicar_filtros(df2025_raw)
df2026_filt, pasos_2026 = aplicar_filtros(df2026_raw)
pasos = pd.concat([pasos_2025, pasos_2026], ignore_index=True)

orden = ["entran", "excl_no_evaluable", "excl_no_cumple", "excluidos", "quedan"]
display(pasos.pivot_table(index="paso", columns="periodo_archivo", values="excluidos", aggfunc="sum").fillna(0).astype(int))

periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
paso,,,,,
1. Edad finita y >= 15,16253,15882,15767,14984,14803
2. Ocupado (OCUPADOS = 1),13062,12942,13443,13121,13306
3. Asalariado (P05C16 en 1-4),8854,8851,8923,8569,8476
4. Salario finito y > 0,0,0,0,0,0
"5. Antigüedad válida (años>=0, meses 0-11 enteros)",0,0,0,0,0
6. Antigüedad <= edad,0,0,0,0,0
"7. Horas habituales en (0, 168]",0,0,0,0,0


In [28]:
# Detalle: tipo de exclusión en cada paso (totales 2025 y 2026)
pasos["conjunto"] = np.where(pasos["periodo_archivo"].str.startswith("2025"), "2025", "2026")
display(pasos.groupby(["conjunto", "paso"])[orden].sum())
pasos.to_csv(OUT_DIR / "resumen_filtros.csv", index=False)

entran  excl_no_evaluable  excl_no_cumple  excluidos  quedan
conjunto paso                                                                                                            
2025     1. Edad finita y >= 15                              203676                  0           62886      62886  140790
         2. Ocupado (OCUPADOS = 1)                           140790                  0           52568      52568   88222
         3. Asalariado (P05C16 en 1-4)                        88222                  0           35197      35197   53025
         4. Salario finito y > 0                              53025                  0               0          0   53025
         5. Antigüedad válida (años>=0, meses 0-11 enteros)   53025                  0               0          0   53025
         6. Antigüedad <= edad                                53025                  0               0          0   53025
         7. Horas habituales en (0, 168]                      53025                  0               0          0   53025
2026     1. Edad finita y >= 15                               49843                  0           14803      14803   35040
         2. Ocupado (OCUPADOS = 1)                            35040                  0           13306      13306   21734
         3. Asalariado (P05C16 en 1-4)                        21734                  0            8476       8476   13258
         4. Salario finito y > 0                              13258                  0               0          0   13258
         5. Antigüedad válida (años>=0, meses 0-11 enteros)   13258                  0               0          0   13258
         6. Antigüedad <= edad                                13258                  0               0          0   13258
         7. Horas habituales en (0, 168]                      13258                  0               0          0   13258

In [29]:
antes = pd.Series({p: m["filas"] for p, m in META.items()}, name="registros_originales")
despues = pd.concat([pasos_2025, pasos_2026]).groupby("periodo_archivo")["quedan"].last().rename("registros_analiticos")
resumen = pd.concat([antes, despues], axis=1)
resumen["% conservado"] = (100 * resumen.registros_analiticos / resumen.registros_originales).round(1)
resumen.loc["Total 2025"] = resumen.loc[PERIODOS_2025].sum()
resumen.loc["Total 2025", "% conservado"] = round(100 * resumen.loc["Total 2025", "registros_analiticos"] / resumen.loc["Total 2025", "registros_originales"], 1)
display(resumen)

,registros_originales,registros_analiticos,% conservado
2025T1,51588.0,13419.0,26.0
2025T2,51167.0,13492.0,26.4
2025T3,51583.0,13450.0,26.1
2025T4,49338.0,12664.0,25.7
2026T1,49843.0,13258.0,26.6
Total 2025,203676.0,53025.0,26.0


**Interpretación de los filtros.** Toda la reducción ocurre en los tres primeros pasos, que **definen la población** en lugar de depurar errores: se excluye a menores de 15 años (paso 1), a quienes no están ocupados (paso 2) y a los ocupados que no son asalariados: cuenta propia, patronos y no remunerados (paso 3). En ninguno de estos pasos hubo registros “no evaluables”: la edad existe para todos.

Los pasos 4 a 7 no excluyeron registros en los archivos procesados: todos los asalariados tienen salario positivo, antigüedad con meses enteros entre 0 y 11, antigüedad menor o igual a la edad y horas entre 1 y 126. Que un filtro excluya cero registros **también es un resultado**: confirma la consistencia de las bases publicadas en estas variables. Aun así, los filtros se aplican siempre y en el mismo orden, para que el procedimiento siga siendo válido con otros archivos (por ejemplo, III 2025). Al final se conserva cerca del 26 % de cada archivo, con tamaños muy parecidos entre trimestres. Los mismos filtros, en el mismo orden, se aplican a 2026.

## 6. Construcción de las variables analíticas y validación de categorías

* `antiguedad = antiguedad_anios + antiguedad_meses / 12` (en años).
* Categóricas: códigos validados contra el diccionario; valores ausentes o no reconocidos → `DESCONOCIDO`. El código educativo `0` (“Ninguno”) se conserva como categoría válida.
* Se añade una columna descriptiva (`*_desc`) con la etiqueta del diccionario para las gráficas.

In [30]:
def mapa(diccionario):
    return F.create_map(*[F.lit(x) for kv in diccionario.items() for x in kv])

def validar(col, catalogo):
    return F.when(F.col(col).isin(list(catalogo.keys())), F.col(col)).otherwise(F.lit("DESCONOCIDO"))

def preparar(df):
    return (df
        .withColumn("salario_mensual", F.col("P05D01"))
        .withColumn("edad", F.col("P02A03"))
        .withColumn("antiguedad_anios", F.col("P05C07A"))
        .withColumn("antiguedad_meses", F.col("P05C07B"))
        .withColumn("antiguedad", F.col("P05C07A") + F.col("P05C07B") / 12)
        .withColumn("horas_semanales", F.col("P05H01A"))
        .withColumn("nivel_educativo", validar("P03A03A", CAT_EDUCACION))
        .withColumn("categoria_ocupacional", validar("P05C16", CAT_OCUPACION))
        .withColumn("dominio", validar("DOMINIO", CAT_DOMINIO))
        .withColumn("ocupado", F.col("OCUPADOS"))
        .withColumn("nivel_educativo_desc", F.coalesce(mapa(CAT_EDUCACION)[F.col("nivel_educativo")], F.lit("DESCONOCIDO")))
        .withColumn("categoria_ocupacional_desc", F.coalesce(mapa(CAT_OCUPACION)[F.col("categoria_ocupacional")], F.lit("DESCONOCIDO")))
        .withColumn("dominio_desc", F.coalesce(mapa(CAT_DOMINIO)[F.col("dominio")], F.lit("DESCONOCIDO")))
        .select(
            "archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario",
            "ANIO", "TRIMESTRE", "NUM_HOGAR", "NUM_PERSONA", "FACTOR",
            "salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "antiguedad", "horas_semanales",
            "nivel_educativo", "nivel_educativo_desc", "categoria_ocupacional", "categoria_ocupacional_desc",
            "dominio", "dominio_desc", "ocupado",
        ))

prep_2025 = preparar(df2025_filt).cache()
prep_2026 = preparar(df2026_filt).cache()

for nombre, d in [("2025", prep_2025), ("2026", prep_2026)]:
    print(f"--- {nombre}: {d.count():,} registros")
    for c in ["nivel_educativo", "categoria_ocupacional", "dominio"]:
        n_desc = d.filter(F.col(c) == "DESCONOCIDO").count()
        print(f"   {c:<22} DESCONOCIDO = {n_desc}")

prep_2025.printSchema()
prep_2025.show(5, truncate=False)

--- 2025: 53,025 registros
   nivel_educativo        DESCONOCIDO = 0
   categoria_ocupacional  DESCONOCIDO = 0
   dominio                DESCONOCIDO = 0
--- 2026: 13,258 registros
   nivel_educativo        DESCONOCIDO = 0
   categoria_ocupacional  DESCONOCIDO = 0
   dominio                DESCONOCIDO = 0
root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: string (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- FACTOR: double (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- antiguedad: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (n

**Interpretación.** En la población analítica no hay códigos ausentes ni no reconocidos en `nivel_educativo`, `categoria_ocupacional` ni `dominio` (0 casos `DESCONOCIDO`). La regla queda implementada de todos modos: un código ausente o fuera del diccionario se etiquetaría `DESCONOCIDO` en lugar de convertirse a `0`, que en el diccionario significa “Ninguno” y cambiaría su interpretación. Los niños de 0 a 6 años, únicos con nivel educativo vacío, ya fueron excluidos por el filtro de edad.

In [31]:
# Unicidad después de filtros
_ = revisar_unicidad(prep_2025.withColumn("hash_fila", F.lit("x")), "2025 preparado")
_ = revisar_unicidad(prep_2026.withColumn("hash_fila", F.lit("x")), "2026 preparado")

2025 preparado: 53,025 registros | 53,025 claves distintas | claves repetidas: 0 | clave nula: 0
2026 preparado: 13,258 registros | 13,258 claves distintas | claves repetidas: 0 | clave nula: 0


## 7. Guardado en Parquet (2025 y 2026 por separado)

In [32]:
RUTA_2025 = OUT_DIR / "eneic_2025_preparado.parquet"
RUTA_2026 = OUT_DIR / "eneic_2026_preparado.parquet"

prep_2025.coalesce(1).write.mode("overwrite").parquet(str(RUTA_2025))
prep_2026.coalesce(1).write.mode("overwrite").parquet(str(RUTA_2026))

for ruta in [RUTA_2025, RUTA_2026]:
    chk = spark.read.parquet(str(ruta))
    print(f"{ruta.name}: {chk.count():,} registros, {len(chk.columns)} columnas")
spark.read.parquet(str(RUTA_2025)).groupBy("periodo_archivo").count().orderBy("periodo_archivo").show()

eneic_2025_preparado.parquet: 53,025 registros, 22 columnas
eneic_2026_preparado.parquet: 13,258 registros, 22 columnas
+---------------+-----+
|periodo_archivo|count|
+---------------+-----+
|         2025T1|13419|
|         2025T2|13492|
|         2025T3|13450|
|         2025T4|12664|
+---------------+-----+



## 8. Respuestas del punto 1

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**
Porque tiene 302 columnas en lugar de 270: el cuestionario cambió (se eliminaron y agregaron preguntas), y las mismas variables quedaron en posiciones diferentes (la edad pasa de la columna 13 a la 9, el salario de la 105 a la 98, `OCUPADOS` de la 266 a la 298). Un apilamiento por posición mezclaría preguntas distintas en una misma columna sin generar error. Por eso se seleccionan las columnas por nombre y se unen con `unionByName` (sección 2.2).

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**
El primero es un faltante **estructural**: el flujo de la boleta no hace esa pregunta a esa persona (el salario no se pregunta a un menor, a un desocupado ni a un trabajador por cuenta propia), así que el valor no existe y la persona queda fuera de la población de estudio. El segundo es **no respuesta**: la pregunta correspondía (asalariado ocupado de 15 años o más), pero no se registró el dato. Solo este segundo tipo implica pérdida de información y posible sesgo. En las bases publicadas todos los vacíos resultaron estructurales (0 % de faltantes entre asalariados de 15+), por lo que el filtro de salario (paso 4) no excluyó registros; en cualquier caso, el salario no se imputa (sección 3.1).

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**
Porque la ENEIC es un panel rotativo: la misma persona entrevistada en dos trimestres genera dos observaciones válidas de su situación laboral en momentos distintos. La unidad de análisis es persona-período, y la clave de unicidad correcta incluye `periodo_archivo` (sección 4). Eliminarla quitaría información real y distorsionaría los conteos y medianas por trimestre. Lo que sí debe reconocerse es que esas observaciones no son independientes.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**
Por tres razones. (1) Es una **muestra**: cada registro representa a muchas personas según su factor de expansión `FACTOR`, que varía entre registros; contar filas sin ponderar no reproduce el total nacional ni su composición (hay grupos sobre o submuestreados por diseño). (2) La base filtrada cubre solo a **asalariados con salario positivo registrado**: excluye a trabajadores por cuenta propia, patronos, no remunerados y a asalariados que no reportaron salario. (3) La unión de 2025 **repite personas** entre trimestres, así que las filas no son personas distintas. Para estimaciones poblacionales habría que usar `FACTOR` como peso (sumando factores para totales y ponderando medias, medianas y proporciones) y, para los errores estándar, el diseño muestral completo (estratos y conglomerados); en este laboratorio los resultados son **no ponderados** y describen únicamente los registros analizados.

In [33]:
# Liberar la caché al finalizar
spark.catalog.clearCache()